In [ ]:
import requests
import pandas as pd
import time
import os

class ApiClient:
    """
    Client per interfacciarsi con l'API degli eventi sismici INGV.
    
    Attributes:
        base_url (str): URL base dell'API.
        default_params (dict): Parametri di default per la query.
        limit (int): Numero massimo di eventi per richiesta.
        timeout (int): Timeout per la richiesta HTTP.
    """
    def __init__(self, base_url, default_params, limit=1000, timeout=60):
        self.base_url = base_url
        self.default_params = default_params.copy()
        self.limit = limit
        self.timeout = timeout

    def fetch_events(self, offset):
        """
        Esegue una richiesta GET all'API per ottenere eventi sismici con offset e limite.
        
        Args:
            offset (int): Offset per la paginazione.
        
        Returns:
            dict: Risposta JSON decodificata dell'API.
        """
        params = self.default_params.copy()
        params.update({
            "offset": offset,
            "limit": self.limit
        })

        response = requests.get(self.base_url, params=params, timeout=self.timeout)
        response.raise_for_status()  # Solleva eccezione in caso di errore HTTP
        return response.json()


class DataManager:
    """
    Gestisce il caricamento e salvataggio dei dati in un file CSV.
    
    Attributes:
        output_file (str): Percorso del file CSV di output.
        records (list): Lista di record già esistenti o appena aggiunti.
    """
    def __init__(self, output_file):
        self.output_file = output_file
        self.records = []
        self._load_existing()

    def _load_existing(self):
        """Carica i dati esistenti dal file CSV, se presente."""
        if os.path.exists(self.output_file):
            df = pd.read_csv(self.output_file)
            self.records = df.to_dict('records')
            print(f"Trovati {len(self.records)} record esistenti")

    def add_records(self, new_records):
        """
        Aggiunge nuovi record alla lista esistente e salva su file.
        
        Args:
            new_records (list): Lista di dizionari contenenti nuovi eventi.
        """
        self.records.extend(new_records)
        self._save()

    def _save(self):
        """Salva tutti i record in un file CSV."""
        df = pd.DataFrame(self.records)
        df.to_csv(self.output_file, index=False)

    @property
    def current_offset(self):
        """
        Calcola l'offset corrente basato sul numero totale di record salvati.
        
        Returns:
            int: Numero di record attualmente presenti.
        """
        return len(self.records)


class Fetcher:
    """
    Coordina il processo di acquisizione degli eventi dall'API e salvataggio su file.
    
    Attributes:
        api_client (ApiClient): Istanza del client API.
        data_manager (DataManager): Gestore per salvataggio dei dati.
        delay (float): Ritardo tra una richiesta e l'altra.
    """
    def __init__(self, output_file, params_base, base_url, 
                 limit=1000, delay=0.5, timeout=60):
        self.api_client = ApiClient(
            base_url=base_url,
            default_params=params_base,
            limit=limit,
            timeout=timeout
        )
        self.data_manager = DataManager(output_file)
        self.delay = delay

    def _process_features(self, features):
        """
        Estrae e normalizza i dati rilevanti da ogni evento.
        
        Args:
            features (list): Lista di eventi (feature) nel formato GeoJSON.
        
        Returns:
            list: Lista di dizionari con le informazioni principali di ogni evento.
        """
        records = []
        for feature in features:
            props = feature['properties']
            geometry = feature['geometry']
            records.append({
                'time': props['time'],
                'magnitude': props['mag'],
                'place': props['place'],
                'depth': geometry['coordinates'][2],
                'latitude': geometry['coordinates'][1],
                'longitude': geometry['coordinates'][0]
            })
        return records

    def run(self):
        """
        Avvia il processo iterativo di fetch e salvataggio degli eventi.
        Continua finché non ci sono più eventi da scaricare.
        """
        offset = self.data_manager.current_offset
        print(f"Inizio acquisizione da offset {offset}")

        while True:
            print(f"Richiesta eventi a offset {offset}...")
            try:
                data = self.api_client.fetch_events(offset)
            except Exception as e:
                print(f"Errore nella richiesta: {e}")
                break

            features = data.get('features', [])
            if not features:
                print("Nessun altro evento trovato")
                break

            new_records = self._process_features(features)
            self.data_manager.add_records(new_records)

            print(f"Salvati {len(new_records)} nuovi eventi " 
                  f"(totale: {self.data_manager.current_offset})")

            offset += self.api_client.limit
            time.sleep(self.delay)

        print("Acquisizione completata con successo")

In [ ]:
# Parametri per la richiesta all'API INGV
PARAMS_BASE = {
    "format": "geojson",
    "starttime": "1985-01-01T00:00:00",
    "endtime": "2025-01-01T00:00:00",
    "minmag": 0,
    "maxmag": 10,
    "mindepth": -10,
    "maxdepth": 1000,
    "orderby": "time-asc",
    "lat": 42.5,
    "lon": 13.5,
    "maxradiuskm": 600
}

# Avvio del processo di fetch degli eventi sismici
fetcher = Fetcher(
    output_file="../data/terremoti_ingv_1985_2025.csv",
    params_base=PARAMS_BASE,
    base_url="https://webservices.ingv.it/fdsnws/event/1/query",
    limit=1000,
    delay=0.5
)

fetcher.run()